## 准备数据

In [8]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [9]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [10]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        self.W1 = tf.Variable(tf.random.normal([784, 256], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([256]))
        self.W2 = tf.Variable(tf.random.normal([256, 10], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([10]))
        ####################
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        x = tf.reshape(x, [-1, 784])
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        logits = tf.matmul(h1, self.W2) + self.b2
        ####################
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [11]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [14]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 1.0936952 ; accuracy 0.7206333
epoch 1 : loss 1.0897434 ; accuracy 0.7218
epoch 2 : loss 1.0858295 ; accuracy 0.72295
epoch 3 : loss 1.0819533 ; accuracy 0.72398335
epoch 4 : loss 1.0781134 ; accuracy 0.72515
epoch 5 : loss 1.0743097 ; accuracy 0.7262667
epoch 6 : loss 1.0705419 ; accuracy 0.72723335
epoch 7 : loss 1.0668089 ; accuracy 0.72826666
epoch 8 : loss 1.0631107 ; accuracy 0.7291
epoch 9 : loss 1.0594467 ; accuracy 0.7303
epoch 10 : loss 1.0558167 ; accuracy 0.7312667
epoch 11 : loss 1.0522205 ; accuracy 0.7320333
epoch 12 : loss 1.0486575 ; accuracy 0.73293334
epoch 13 : loss 1.0451269 ; accuracy 0.73395
epoch 14 : loss 1.0416286 ; accuracy 0.73475
epoch 15 : loss 1.038162 ; accuracy 0.73565
epoch 16 : loss 1.0347266 ; accuracy 0.7364333
epoch 17 : loss 1.0313216 ; accuracy 0.73726666
epoch 18 : loss 1.0279474 ; accuracy 0.7381167
epoch 19 : loss 1.0246031 ; accuracy 0.7389
epoch 20 : loss 1.0212885 ; accuracy 0.73975
epoch 21 : loss 1.0180033 ; accuracy 0.7405